# Pipeline Pan-Genome Bacterien

**Auteur : Marwa Zidi** — Universite Paris Cite

---

## Fonctionnement

- `______` : a remplacer par vos valeurs (noms de fichiers, variables...)
- Cellules deja remplies : executer directement
- Ne faites pas de copier-coller : ecrivez les commandes vous-meme

## Organisation des donnees

| Dossier | Contenu |
|---------|--------|
| `/root/data/for-Pangenome /Raw_Files/` | Reads FASTQ bruts (R1/R2) |
| `/root/data/for-Pangenome /Inputs/` | Assemblages pre-calcules |
| `/root/results/` | Vos resultats |

---

# PHASE 1 — Controle qualite et Assemblage

**Objectifs** :
1. Verifier la qualite des reads (FastQC)
2. Nettoyer les adaptateurs et bases de mauvaise qualite (Trimmomatic)
3. Assembler les reads en contigs (SPAdes)
4. Evaluer la qualite de l'assemblage (QUAST)

---

## Etape 1.1 — Explorer les donnees disponibles

Commencez par identifier les fichiers disponibles avant d'ecrire vos commandes.

In [ ]:
# Lister les fichiers FASTQ bruts
ls -lh /root/data/for-Pangenome\ /Raw_Files/

In [ ]:
# Lister les assemblages disponibles
ls -lh /root/data/for-Pangenome\ /Inputs/

In [ ]:
# Creer les repertoires de travail (commande fournie)
mkdir -p /root/results/01_quality_control
mkdir -p /root/results/02_trimmed
mkdir -p /root/results/03_assembly
mkdir -p /root/results/04_annotation
mkdir -p /root/results/05_pangenome
echo "Repertoires crees"

## Etape 1.2 — Controle qualite avec FastQC

**Consigne** : Remplacez les `______` par vos noms de fichiers.

Commencez par lister les fichiers disponibles (etape 1.1) pour copier les noms exacts.

In [ ]:
# Definir les variables — remplacez les ______
SAMPLE=______          # Nom de votre souche (ex: strain1)

R1="/root/data/for-Pangenome /Raw_Files/______"   # Fichier R1
R2="/root/data/for-Pangenome /Raw_Files/______"   # Fichier R2

OUTPUT_DIR="/root/results/01_quality_control"

echo "Echantillon : $SAMPLE"
echo "R1 : $R1"
echo "R2 : $R2"

In [ ]:
# Lancer FastQC — remplacez les ______ par les variables R1 et R2
fastqc -o $OUTPUT_DIR ______ ______

echo "Rapports FastQC generes dans $OUTPUT_DIR"

**Question** : Ouvrez les rapports HTML generes. La qualite des reads vous semble-t-elle suffisante pour l'assemblage ?

*Indicateurs a verifier* : Per base sequence quality, GC content, Adapter content

## Etape 1.3 — Nettoyage avec Trimmomatic

Trimmomatic supprime les adaptateurs et les bases de mauvaise qualite.

**Parametres utilises** :
- `LEADING:3 TRAILING:3` : supprime les bases de debut/fin si qualite < 3
- `SLIDINGWINDOW:4:15` : fenetre glissante de 4 bases, qualite moyenne >= 15
- `MINLEN:36` : supprime les reads de moins de 36 pb

In [ ]:
# Definir les chemins de sortie — completez les suffixes ______
R1_OUT="/root/results/02_trimmed/${SAMPLE}______"       # ex: _R1_paired.fastq.gz
R1_UNPAIRED="/root/results/02_trimmed/${SAMPLE}______"
R2_OUT="/root/results/02_trimmed/${SAMPLE}______"
R2_UNPAIRED="/root/results/02_trimmed/${SAMPLE}______"

echo "R1 paired   : $R1_OUT"
echo "R2 paired   : $R2_OUT"

In [ ]:
# Lancer Trimmomatic — completez les 6 blancs (entrees/sorties)
trimmomatic PE -phred33 \
    ______ ______ \
    ______ ______ \
    ______ ______ \
    ILLUMINACLIP:/usr/share/trimmomatic/NexteraPE-PE.fa:2:30:10 \
    LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

echo "Reads nettoyes"

## Etape 1.4 — Assemblage avec SPAdes

> **Attention** : SPAdes peut prendre 30-60 minutes. Un assemblage pre-calcule est disponible dans
> `/root/data/for-Pangenome /Inputs/` si vous souhaitez passer directement a l'etape suivante.

In [ ]:
# Definir les chemins — completez les suffixes ______
R1_CLEAN="/root/results/02_trimmed/${SAMPLE}______"   # fichier R1 paired
R2_CLEAN="/root/results/02_trimmed/${SAMPLE}______"   # fichier R2 paired
OUTPUT_DIR="/root/results/03_assembly/$SAMPLE"

echo "Assemblage de : $SAMPLE"
echo "R1 nettoye : $R1_CLEAN"
echo "R2 nettoye : $R2_CLEAN"

In [ ]:
# Lancer SPAdes — completez les 3 blancs
spades.py -1 ______ -2 ______ \
    -o ______ \
    --careful \
    -t 4

echo "Assemblage termine : ${OUTPUT_DIR}/contigs.fasta"

## Etape 1.5 — Validation avec QUAST

QUAST calcule les metriques de qualite de l'assemblage : N50, nombre de contigs, genome total...

In [ ]:
# Definir les chemins — completez le nom du fichier
CONTIGS="/root/results/03_assembly/$SAMPLE/______"   # ex: contigs.fasta
OUTPUT_DIR="/root/results/03_assembly/${SAMPLE}_quast"

echo "Validation de : $CONTIGS"

In [ ]:
# Lancer QUAST — completez les 2 blancs
quast.py ______ -o ______

echo "Rapport QUAST dans ${OUTPUT_DIR}/report.html"

In [ ]:
# Afficher les metriques principales
cat ${OUTPUT_DIR}/report.txt

**Question** : Quels sont le N50 et le nombre de contigs de votre assemblage ? Ces valeurs vous semblent-elles satisfaisantes ?

---

# PHASE 2 — Identification et Annotation

---

## Etape 2.1 — Typage MLST avec pubMLST (outil web)

1. Aller sur https://pubmlst.org/
2. Selectionner votre espece bacterienne
3. Uploader votre fichier `contigs.fasta` : `/root/results/03_assembly/______/contigs.fasta`
4. Lancer l'analyse et noter le Sequence Type (ST) obtenu
5. Sauvegarder les resultats dans `/root/results/04_annotation/`

**Resultat attendu** : Sequence Type (ST), alleles identifies, clonal complex

## Etape 2.2 — Annotation avec Prokka

Prokka annote automatiquement un genome bacterien et produit les fichiers GFF necessaires pour Roary.

> Les fichiers **GFF sont essentiels** pour la Phase 3 !

In [ ]:
# Definir les chemins et l'espece — completez les ______
CONTIGS="/root/results/03_assembly/$SAMPLE/______"   # contigs.fasta
OUTPUT_DIR="/root/results/04_annotation/${SAMPLE}_prokka"

GENUS=______    # Nom du genre  (ex: Escherichia)
SPECIES=______  # Nom de l'espece (ex: coli)

echo "Annotation de : $SAMPLE ($GENUS $SPECIES)"

In [ ]:
# Lancer Prokka — completez les 7 blancs
prokka --outdir ______ \
    --prefix ______ \
    --kingdom Bacteria \
    --genus ______ \
    --species ______ \
    --strain ______ \
    --cpus 4 \
    --force \
    ______

echo "Fichier GFF : ${OUTPUT_DIR}/${SAMPLE}.gff"

In [ ]:
# Afficher les statistiques d'annotation
cat ${OUTPUT_DIR}/${SAMPLE}.txt

**Fichiers Prokka generes** :
- `SAMPLE.gff` : annotation complete — **REQUIS pour Roary**
- `SAMPLE.gbk` : format GenBank — pour antiSMASH
- `SAMPLE.faa` : sequences proteiques
- `SAMPLE.txt` : statistiques (nombre de CDS, ARNr, ARNt...)

## Etape 2.3 — Metabolites secondaires avec antiSMASH (outil web)

1. Aller sur https://antismash.secondarymetabolites.org/
2. Uploader le fichier `.gbk` de Prokka : `/root/results/04_annotation/______/______.gbk`
3. Cocher : KnownClusterBlast, ClusterBlast, SubClusterBlast
4. Lancer l'analyse (10-30 minutes)
5. Telecharger les resultats dans `/root/results/04_annotation/`

**Resultat attendu** : clusters NRPS, PKS, bacteriocines, terpenes...

## Etape 2.4 — Enrichissement fonctionnel avec enrichR (optionnel)

In [ ]:
# Extraire les genes de resistance depuis le GFF
GFF_FILE="/root/results/04_annotation/${SAMPLE}_prokka/${SAMPLE}.gff"
GENES_FILE="/root/results/04_annotation/genes_resistance.txt"

grep -i "resistance" $GFF_FILE | cut -f9 | cut -d';' -f1 | sed 's/ID=//g' > $GENES_FILE

echo "Genes de resistance extraits :"
head $GENES_FILE

In [ ]:
# Lancer enrichR — completez le chemin du fichier de genes
bash /root/enrichr_shell.sh -f ______ -o /root/results/04_annotation/enrichr_results

echo "Resultats dans enrichr_results/enrichr_results.csv"

---

# PHASE 3 — Pan-Genome

---

## Etape 3.1 — Telecharger des genomes de reference (NCBI)

Roary compare plusieurs genomes : vous avez besoin d'au moins **3 fichiers GFF**.

1. Aller sur https://www.ncbi.nlm.nih.gov/genome/
2. Rechercher votre espece bacterienne
3. Selectionner 3 a 10 genomes representatifs
4. Telecharger les fichiers FASTA ou GFF
5. Sauvegarder dans `/root/results/05_pangenome/reference_genomes/`

> Si vous telechargez des FASTA, annotez-les avec Prokka (etape ci-dessous) pour obtenir les GFF.

In [ ]:
mkdir -p /root/results/05_pangenome/reference_genomes

In [ ]:
# Annoter un genome de reference NCBI (si vous avez un FASTA)
# Completez les 7 blancs

REF_GENOME="/root/results/05_pangenome/reference_genomes/______"   # fichier FASTA
REF_NAME=______   # nom court (ex: K12_reference)
OUTPUT_DIR="/root/results/05_pangenome/reference_genomes/${REF_NAME}_prokka"

prokka --outdir ______ \
    --prefix ______ \
    --kingdom Bacteria \
    --genus ______ \
    --species ______ \
    --cpus 4 \
    --force \
    ______

echo "GFF de reference : ${OUTPUT_DIR}/${REF_NAME}.gff"

## Etape 3.2 — Preparer les GFF pour Roary

Tous les fichiers GFF doivent etre reunis dans un meme dossier.

> **Minimum requis : 3 fichiers GFF**

In [ ]:
mkdir -p /root/results/05_pangenome/all_gffs

In [ ]:
# Copier vos GFF — completez les noms des dossiers et fichiers

# Souche 1
cp /root/results/04_annotation/______/______.gff /root/results/05_pangenome/all_gffs/

# Souche 2
cp /root/results/04_annotation/______/______.gff /root/results/05_pangenome/all_gffs/

# Souche 3
cp /root/results/04_annotation/______/______.gff /root/results/05_pangenome/all_gffs/

echo "GFF copies"

In [ ]:
# Copier les GFF de reference (si annotes avec Prokka)
cp /root/results/05_pangenome/reference_genomes/*_prokka/*.gff /root/results/05_pangenome/all_gffs/
echo "GFF de reference copies"

In [ ]:
# Verifier le contenu et le nombre de GFF
ls -lh /root/results/05_pangenome/all_gffs/

NB_GFF=$(ls /root/results/05_pangenome/all_gffs/*.gff 2>/dev/null | wc -l)
echo "Nombre de fichiers GFF : $NB_GFF"

if [ $NB_GFF -lt 3 ]; then
    echo "ATTENTION : Roary necessite au moins 3 genomes !"
else
    echo "Nombre de GFF suffisant pour Roary"
fi

## Etape 3.3 — Calcul du pan-genome avec Roary

Roary identifie les genes partages (core genome) et les genes variables (accessory genome).

**Arguments utilises** :
- `-e -n` : alignement des genes core avec MAFFT
- `-p 4` : 4 processeurs en parallele
- `-v` : mode verbeux

In [ ]:
GFF_DIR="/root/results/05_pangenome/all_gffs"
OUTPUT_DIR="/root/results/05_pangenome/roary_output"

echo "GFF disponibles : $(ls $GFF_DIR/*.gff | wc -l)"
echo "Sortie : $OUTPUT_DIR"

In [ ]:
# Lancer Roary — completez les 2 blancs
roary -f ______ \
    -e -n \
    -p 4 \
    -v \
    ______/*.gff

echo "Analyse pan-genome terminee"

In [ ]:
# Afficher les statistiques du pan-genome
cat ${OUTPUT_DIR}/summary_statistics.txt

In [ ]:
# Apercu de la matrice presence/absence
head -n 5 ${OUTPUT_DIR}/gene_presence_absence.csv | cut -d',' -f1-5

**Questions** :
- Combien de genes sont dans le **core genome** (presents dans tous les genomes) ?
- Combien de genes sont dans le **soft core** (95-99% des genomes) ?
- Combien de genes sont **specifiques** a une seule souche ?
- Que nous apprend cette repartition sur la diversite de votre espece ?

## Etape 3.4 — Visualisation avec Phandango (outil web)

1. Aller sur https://jameshadfield.github.io/phandango/
2. Uploader :
   - **Arbre** : `results/05_pangenome/roary_output/accessory_binary_genes.fa.newick`
   - **Matrice** : `results/05_pangenome/roary_output/gene_presence_absence.csv`
3. Explorer la visualisation : zoom, filtres core/accessoire, cliquer sur les genes
4. Sauvegarder en SVG ou PNG

**Question** : Identifiez les souches les plus proches genetiquement. Sont-elles celles que vous attendiez ?

---

## Checklist finale

**Phase 1 — Assemblage**
- [ ] FastQC execute et rapports examines
- [ ] Trimmomatic : reads nettoyes
- [ ] SPAdes : assemblage genere
- [ ] QUAST : qualite validee (N50, nb contigs)

**Phase 2 — Annotation**
- [ ] pubMLST : Sequence Type identifie
- [ ] Prokka : fichiers GFF generes
- [ ] antiSMASH : clusters de metabolites identifies

**Phase 3 — Pan-genome**
- [ ] Au moins 3 fichiers GFF reunis
- [ ] Roary : pan-genome calcule
- [ ] Phandango : visualisation completee

---

**Felicitations ! Vous avez complete le pipeline pan-genome.**